In [23]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms
from torchvision.models import convnext_tiny
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import matplotlib.pyplot as plt
from efficient_kan import KAN

In [39]:
def get_baseline(num_classes=5):
    model = convnext_tiny(weights=None) 
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    return model

class ConvNeXtKAN(nn.Module):
    def __init__(self, num_classes=5, kan_hidden_dim=32):
        super().__init__()
        self.backbone = convnext_tiny(weights=None)
        self.backbone.classifier[2] = KAN([self.backbone.classifier[2].in_features, kan_hidden_dim, num_classes])
        
    def forward(self, x):
        return self.backbone(x)

    def get_regularization_loss(self):
        return self.backbone.classifier[2].regularization_loss()

In [40]:
# def get_baseline(num_classes=5, dropout_p=0.5):
#     model = models.convnext_tiny(weights=None) 
#     in_features = model.classifier[2].in_features

#     model.classifier[2] = torch.nn.Sequential(
#         torch.nn.Dropout(p=dropout_p),
#         torch.nn.Linear(in_features, num_classes)
#     )
#     return model

# class ConvNeXtKAN(nn.Module):
#     def __init__(self, num_classes=5, kan_hidden_dim=32, dropout_p=0.5):
#         super(ConvNeXtKAN, self).__init__()

#         weights = ConvNeXt_Tiny_Weights.DEFAULT
#         self.backbone = convnext_tiny(weights=weights)
#         in_features = self.backbone.classifier[2].in_features
#         # self.backbone.classifier[2] = KAN([in_features, kan_hidden_dim, num_classes])

#         self.classifier_head = nn.Sequential(
#             nn.Dropout(p=dropout_p),
#             KAN([in_features, kan_hidden_dim, num_classes])
#         )

#         self.backbone.classifier[2] = self.classifier_head

#     def forward(self, x):
#         return self.backbone(x)

#     def get_regularization_loss(self):
#         return self.backbone.classifier[2][1].regularization_loss()

In [41]:
def process_and_save_cam(model, target_layer, img_path, save_path, device):
    """Loads a single image, processes the Grad-CAM, and saves the output directly."""
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # get image tensor
    raw_img = Image.open(img_path).convert('RGB')
    input_tensor = transform(raw_img).unsqueeze(0).to(device)
    
    # Generate Heatmap
    cam = GradCAM(model=model, target_layers=[target_layer])
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]
    
    img_np = transform(raw_img).permute(1, 2, 0).numpy()
    img_np = np.clip(np.array([0.229, 0.224, 0.225]) * img_np + np.array([0.485, 0.456, 0.406]), 0, 1)
    
    # save
    visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    plt.imsave(save_path, visualization)


In [42]:
def build_deterministic_image_dict(csv_path, raw_dir):
    """Parses the master CSV to find exactly one image per class for both domains."""
    df = pd.read_csv(csv_path)
    
    target_classes = ['MEL', 'NV', 'BCC', 'BKL', 'AKIEC']
    class_col = 'label'       
    img_col = 'image_id'         
    dataset_col = 'domain'    
    
    # Map csv to local folder
    folder_map = {
        'HAM10000': 'HAM10000',
        'PAD_UFES_20': 'PAD_UFES_20'
    }
    
    target_images = {}

    # Get 5 images from HAM
    source_df = df[df[dataset_col] == 'HAM10000']
    for cls in target_classes:
        row = source_df[source_df[class_col] == cls].iloc[0]

        filename = row[img_col] if str(row[img_col]).endswith(('.jpg', '.png')) else f"{row[img_col]}.jpg"
        dataset_folder = folder_map[row[dataset_col]]

        full_path = os.path.join(raw_dir, dataset_folder, filename)
        target_images[f"HAM_{cls}"] = full_path
        
    # Get 5 images from PAD
    target_df = df[df[dataset_col] == 'PAD_UFES_20']
    for cls in target_classes:
        row = target_df[target_df[class_col] == cls].iloc[0]
        
        filename = row[img_col] if str(row[img_col]).endswith(('.jpg', '.png')) else f"{row[img_col]}.png"
        dataset_folder = folder_map[row[dataset_col]]
        
        full_path = os.path.join(raw_dir, dataset_folder, filename)
        target_images[f"PAD_{cls}"] = full_path

    return target_images

csv_path = "./data/raw/master_metadata_split.csv" 
raw_dir = "./data/raw/"                           

target_images = build_deterministic_image_dict(csv_path, raw_dir)

for key, path in target_images.items():
    print(f"{key}: {path}")

HAM_MEL: ./data/raw/HAM10000/ISIC_0031741.jpg
HAM_NV: ./data/raw/HAM10000/ISIC_0030871.jpg
HAM_BCC: ./data/raw/HAM10000/ISIC_0025260.jpg
HAM_BKL: ./data/raw/HAM10000/ISIC_0034003.jpg
HAM_AKIEC: ./data/raw/HAM10000/ISIC_0028558.jpg
PAD_MEL: ./data/raw/PAD_UFES_20/PAT_680_1289_182.png
PAD_NV: ./data/raw/PAD_UFES_20/PAT_1516_1765_530.png
PAD_BCC: ./data/raw/PAD_UFES_20/PAT_46_881_939.png
PAD_BKL: ./data/raw/PAD_UFES_20/PAT_1549_1882_230.png
PAD_AKIEC: ./data/raw/PAD_UFES_20/PAT_1545_1867_547.png


In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on: {device}")

# Final 4 models
models = {
    "Base_Reg":   {"path": "VAULT/Final_4/Paths/baseline_exp1_std_reg_best_loss.pth", "is_kcn": False},
    "Base_Uncon": {"path": "VAULT/Final_4/Paths/baseline_Uncon_best_loss.pth", "is_kcn": False},
    "KCN_Heavy":  {"path": "VAULT/Final_4/Paths/kcn_heavy_spline_best_loss.pth", "is_kcn": True},
    "KCN_Uncon":  {"path": "VAULT/Final_4/Paths/kcn_Uncon_best_loss.pth", "is_kcn": True}
}

base_output_dir = "VAULT/Final_4/Grad-CAD"


for model_name, config in models.items():
    print(f"\nEvaluating Model: {model_name}")
    
    # load weights
    model = ConvNeXtKAN() if config["is_kcn"] else get_baseline()
    model.load_state_dict(torch.load(config["path"], map_location=device), strict=False)
    model = model.eval().to(device)

    target_layer = model.backbone.features[-1] if config["is_kcn"] else model.features[-1]

    for img_name, img_path in target_images.items():
        if not os.path.exists(img_path):
            print(f"  [!] Missing file: {img_path}")
            continue
            
        domain_prefix, cls = img_name.split('_', 1)
        split_folder = "val" if domain_prefix == "HAM" else "test"

        target_dir = os.path.join(base_output_dir, cls, split_folder)
        os.makedirs(target_dir, exist_ok=True)
            
        # Compile the final save path
        save_path = os.path.join(target_dir, f"{model_name}_{img_name}.png")
        
        # Generate and save
        process_and_save_cam(model, target_layer, img_path, save_path, device)
        print(f"  -> Saved {model_name} heatmap to {cls}/{split_folder}/")

print("\n--- All heatmaps generated successfully. ---")

Executing on: cpu

Evaluating Model: Base_Reg
  -> Saved Base_Reg heatmap to MEL/val/
  -> Saved Base_Reg heatmap to NV/val/
  -> Saved Base_Reg heatmap to BCC/val/
  -> Saved Base_Reg heatmap to BKL/val/
  -> Saved Base_Reg heatmap to AKIEC/val/
  -> Saved Base_Reg heatmap to MEL/test/
  -> Saved Base_Reg heatmap to NV/test/
  -> Saved Base_Reg heatmap to BCC/test/
  -> Saved Base_Reg heatmap to BKL/test/
  -> Saved Base_Reg heatmap to AKIEC/test/

Evaluating Model: Base_Uncon
  -> Saved Base_Uncon heatmap to MEL/val/
  -> Saved Base_Uncon heatmap to NV/val/
  -> Saved Base_Uncon heatmap to BCC/val/
  -> Saved Base_Uncon heatmap to BKL/val/
  -> Saved Base_Uncon heatmap to AKIEC/val/
  -> Saved Base_Uncon heatmap to MEL/test/
  -> Saved Base_Uncon heatmap to NV/test/
  -> Saved Base_Uncon heatmap to BCC/test/
  -> Saved Base_Uncon heatmap to BKL/test/
  -> Saved Base_Uncon heatmap to AKIEC/test/

Evaluating Model: KCN_Heavy
  -> Saved KCN_Heavy heatmap to MEL/val/
  -> Saved KCN_Heavy